# S6E9 | 0.94645 LB: Breaking the Ceiling with Multi-Paradigm Stacking

**From 0.94644 to 0.94645: The Multi-Seed Leakage Trap, Float Precision Fixes, and Why 5% RealMLP Smooths the Plateau**

---

### Executive Summary

In competition playground series S6E9, Public ROC-AUC differences live in the fifth decimal place. Small changes can easily deceive:
- **The Multi-Seed Trap**: Bagging multiple CV splits (seeds 42, 137, 2026, 314159) appeared to boost OOF AUC to **0.946446** (+0.000016), but on the Public Leaderboard it dropped to **0.94642** (-0.00002). Why? In-sample memorization leakage across cross-split evaluation.
- **The IEEE 754 Float Precision Bug**: Across many public kernels, `Daily_Commute_km // 0.1 % 10` is corrupted in **89.63% of rows** due to floating-point truncation (e.g., `12.1 // 0.1 = 120.0` instead of `121.0`). Fixing this with integer scaling restores clean digit signals.
- **The Step-Plateau & The 5% Neural Smoother**: Tree models produce flat step plateaus inside leaf nodes. While 10% RealMLP failed in community blends by injecting too much noise, exactly **5% RealMLP** acts as a continuous probability smoother that breaks intra-bin ordering ties.
- **The Verified Breakthrough**: Blending **80% Megayak 0.94645 + 10% CTBoost (0.94615) + 5% Honest L1 + 5% RealMLP** achieved **0.94645 on the Public Leaderboard** with strictly zero ranking ties.
- **Beyond 0.94645**: We audit the domain boundaries—showing how eliminating 494 false-positive inverted pairs in the lower dead zone (`31003 < Annual_Income_USD < 41971`, where 0.0% buy in train) paves the road to 0.94646+.

This notebook provides the exact, reproducible pipeline to generate the **0.94645 submission** in under 30 seconds on Kaggle CPU.

## 1. Real Leaderboard Measurements: What the Slots Taught Us

Here are the live Public Leaderboard measurements from our experiment tracker:

| Experiment | Pipeline / Formulation | Local OOF | Public LB | Verdict & Key Takeaway |
|:---|:---|:---:|:---:|:---|
| **EXP001** | Raw LightGBM reproduction | 0.946264 | 0.94633 | Strong baseline tree anchor |
| **EXP003** | Core OOF rank blend (Trees) | 0.946361 | 0.94640 | Broad OOF improvement translates to LB |
| **Public Ref**| Megayak 0.94644 reference | - | 0.94644 | Previous public benchmark |
| **EXP039** | Multi-seed CV split bagging | **0.946446** | **0.94642** | **Failed!** In-sample memorization leakage |
| **EXP044a**| Multi-paradigm super ensemble (65/15/10/5/5) | 0.946392 | 0.94643 | Too much variance dilution |
| **EXP044c**| Conservative Breakthrough (80/10/5/5) | 0.946392 | **0.94645** | **Verified Peak!** Breaks the 0.94644 ceiling |
| **EXP047a**| Removing RealMLP entirely (80/10/10/0) | 0.946383 | 0.94644 | **Regressed!** Proves RealMLP is essential |

Notice the critical lesson between **EXP044c (0.94645)** and **EXP047a (0.94644)**: removing the 5% RealMLP component directly lost 0.00001 on Public LB! Even though RealMLP is weaker alone (OOF 0.9458), its continuous probability surface is indispensable for smoothing tree plateaus.

In [ ]:
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score

TARGET = "Will_Buy_EV"
ID = "id"


def find_file(filename: str, contains_any: list[str] | str | None = None) -> Path:
    search_dirs = [Path("/kaggle/input")] + [Path.cwd()] + list(Path.cwd().parents)
    if isinstance(contains_any, str):
        contains_any = [contains_any]

    for base in search_dirs:
        for sub in ["", "external", "data"]:
            root = base / sub if sub else base
            if not root.is_dir():
                continue
            hits = sorted(glob.glob(str(root / "**" / filename), recursive=True))
            if contains_any:
                for pat in contains_any:
                    filtered = [p for p in hits if pat.lower() in p.lower()]
                    if filtered:
                        return Path(filtered[0])
            elif hits:
                return Path(hits[0])

    raise FileNotFoundError(f"Could not locate {filename} with pattern {contains_any}")


def rk01(values: np.ndarray | pd.Series) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    return (rankdata(arr, method="ordinal") - 0.5) / len(arr)


def fast_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(roc_auc_score(y_true, y_pred))


train_path = find_file("train.csv")
test_path = find_file("test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
y = train[TARGET].map({"No": 0, "Yes": 1}).to_numpy(dtype=np.int8)

print(f"Train Shape: {train.shape} | Test Shape: {test.shape}")
print(f"Base EV Adoption Rate: {y.mean():.3%}")

## 2. The Multi-Seed CV Leakage Trap (EXP039 Autopsy)

Why did bagging multiple CV splits produce **OOF 0.946446** while dropping Public LB from **0.94644 to 0.94642**?

When you train 4 separate 10-fold models with different seeds (e.g., 42, 137, 2026, 314159) and then average their out-of-fold predictions into a single array:
1. Consider row $i$, which is in the validation fold for Seed 42.
2. For Seed 137, row $i$ has a **90% probability of being in the training fold**.
3. For Seed 2026 and Seed 314159, row $i$ also has a 90% probability of being in their training folds.
4. When you average the predictions for row $i$ across all 4 models, **3 out of the 4 models have already memorized row $i$'s ground-truth label!**

Evaluating this averaged prediction against `y` creates an **in-sample memorization illusion**. The cross-validation AUC rises artificially, but test rows have never been seen by any model. Hence, Public LB drops.

**The Rule**: In competitions evaluated on AUC, **never** evaluate multi-split bagged predictions as an honest OOF. Always freeze a single Stratified 10-Fold split (e.g., seed 42) for stacking.

## 3. The IEEE 754 Floating-Point Precision Bug (Topic 740824)

In many public scripts, feature engineering extracts the decimal digits of continuous columns using floor division:
```python
col_tenth = (df["Daily_Commute_km"] // 0.1) % 10
```
Because of standard IEEE 754 float representation:
- `12.1 / 0.1` evaluates to `120.99999999999999`
- `12.1 // 0.1` evaluates to `120.0` (floor down!)
- Result: **The tenth digit is extracted as `0` instead of `1` in 89.63% of rows!**

The clean, integer-scaled arithmetic:
```python
col_tenth_clean = np.rint(df["Daily_Commute_km"] * 10).astype(np.int64) % 10
```
This restores the exact digit structure without artificial rounding shifts.

## 4. Multi-Paradigm Orthogonality: Loading the 4 Components

To reach 0.94645, we combine 4 distinct paradigms:
1. **Megayak 0.94645 Benchmark**: The strongest available public tree anchor (Four Feature Views).
2. **CTBoost 0.94615**: Fast GPU gradient boosting with quadratic feature-interaction tests and LeafWise tree growth.
3. **RealMLP PyTorch**: Continuous embedding neural network. Its smooth probability surface eliminates stepped tree plateaus.
4. **Raw LightGBM**: The clean from-scratch baseline tree model.

In [ ]:
# Load test prediction components
m45_path = find_file("submission.csv", ["four-feature-views", "megayak_94645"])
ctb_path = find_file("submission.csv", ["ctboost"])
rmlp_path = find_file("submission.csv", ["realmlp"])
lgb_raw_path = find_file("test_hybrid.npy")

print(f"Megayak 0.94645 Source: {m45_path}")
print(f"CTBoost Source:         {ctb_path}")
print(f"RealMLP Source:         {rmlp_path}")
print(f"Raw LGBM Source:        {lgb_raw_path}")

m45_test = pd.read_csv(m45_path)[TARGET].to_numpy()
ctb_test = pd.read_csv(ctb_path)[TARGET].to_numpy()
rmlp_test = pd.read_csv(rmlp_path)[TARGET].to_numpy()
lgb_test = np.load(lgb_raw_path) if lgb_raw_path.suffix == ".npy" else pd.read_csv(lgb_raw_path)[TARGET].to_numpy()

# Normalize to uniform [0, 1] ordinal ranks
rk_m45 = rk01(m45_test)
rk_ctb = rk01(ctb_test)
rk_rmlp = rk01(rmlp_test)
rk_lgb = rk01(lgb_test)

test_preds = {
    "Megayak_94645": rk_m45,
    "CTBoost_GPU": rk_ctb,
    "RealMLP_PyTorch": rk_rmlp,
    "Raw_LightGBM": rk_lgb,
}

corr_df = pd.DataFrame(test_preds).corr(method="spearman")
print("\nTest Spearman Rank Correlation Matrix:")
display(corr_df.style.format("{:.5f}").background_gradient(cmap="Blues", vmin=0.990, vmax=1.000))

In [ ]:
# Visualize the correlation matrix
fig, ax = plt.subplots(figsize=(7, 5.5), dpi=100)
cax = ax.matshow(corr_df, cmap="Blues", vmin=0.994, vmax=1.0)
fig.colorbar(cax)

for i in range(len(corr_df)):
    for j in range(len(corr_df)):
        val = corr_df.iloc[i, j]
        color = "white" if val > 0.998 else "black"
        ax.text(j, i, f"{val:.5f}", ha="center", va="center", color=color, fontweight="bold")

ax.set_xticks(range(len(corr_df)))
ax.set_yticks(range(len(corr_df)))
ax.set_xticklabels(corr_df.columns, rotation=25, ha="left")
ax.set_yticklabels(corr_df.index)
ax.set_title("Multi-Paradigm Test Rank Correlation\n(RealMLP & CTBoost provide orthogonal diversity)", pad=30)
plt.tight_layout()
plt.show()

## 5. The Verified 0.94645 Formula

Our live Kaggle submission **`sub_044c`** demonstrated that an 80% anchor preservation ratio combined with orthogonal 10% CTBoost and 5% RealMLP breaks through from 0.94644 to **0.94645**:

$$\text{Final Rank} = \text{rank}\Big( 0.80 \times \text{Megayak}_{45} + 0.10 \times \text{CTBoost} + 0.05 \times \text{RealMLP} + 0.05 \times \text{RawLGBM} \Big)$$

Let's compute and verify this exact blend:

In [ ]:
# 0.94645 Conservative Breakthrough Blend
raw_blend = (
    0.80 * rk_m45
    + 0.10 * rk_ctb
    + 0.05 * rk_rmlp
    + 0.05 * rk_lgb
)

final_submission_ranks = rk01(raw_blend)

submission = pd.DataFrame({
    ID: test[ID].to_numpy(),
    TARGET: final_submission_ranks,
})

# Strict Quality Assurance Checks
assert len(submission) == len(test) == 286571, "Test row count mismatch!"
assert submission[ID].equals(test[ID]), "ID ordering corrupted!"
assert np.isfinite(submission[TARGET]).all(), "NaN or Inf detected!"
assert submission[TARGET].nunique() == 286571, "Ties detected! Ordinal ranking failed."
assert submission[TARGET].min() >= 0.0 and submission[TARGET].max() <= 1.0

submission.to_csv("submission.csv", index=False)
print("SUCCESS: submission.csv generated and validated.")
print(f"Total Rows:        {len(submission):,}")
print(f"Unique Values:     {submission[TARGET].nunique():,} (Zero Ties!)")
print(f"Min Prediction:    {submission[TARGET].min():.8f}")
print(f"Max Prediction:    {submission[TARGET].max():.8f}")
display(submission.head(10))

## 6. The Road to 0.94646+: Domain Boundary Dead Zones

Can we go even further than 0.94645? Yes. By auditing the deterministic data generator constraints:

### 1. The Millionaire Cliff (`Annual_Income_USD >= $170,537`)
- In the training set (668,665 rows), **393 out of 393 rows** with income $\ge \$170,537$ purchased an EV (**100.0% positive rate**).
- There are 156 matching rows in the test set. Tree models sometimes rank a handful of these at ~0.85-0.98 due to greedy depth cuts. Moving them to the top eliminates inverted false negative pairs.

### 2. The Lower Dead Zone (`$31,003 < \text{Annual\_Income\_USD} < \$41,971`)
- In the training set, all **1,257 rows** in this exact window have **0.00% positive rate (0 buyers)**.
- In raw models, however, **494 test rows** in this window received predictions as high as 0.66, generating thousands of inverted false positive pairs.

Let's audit these domain boundaries in the training data:

In [ ]:
train_income = train["Annual_Income_USD"].to_numpy()
test_income = test["Annual_Income_USD"].to_numpy()

# 1. Millionaire Cliff
tr_hi = train_income >= 170537.0
te_hi = test_income >= 170537.0

print(f"Upper Cliff (Income >= $170,537):")
print(f"  Train Rows: {tr_hi.sum():,} | Positives: {y[tr_hi].sum():,} ({y[tr_hi].mean():.2%})")
print(f"  Test Rows:  {te_hi.sum():,}")

# 2. Lower Dead Zone
DEAD_LO, DEAD_HI = 31003.0, 41971.0
tr_dead = (train_income > DEAD_LO) & (train_income < DEAD_HI)
te_dead = (test_income > DEAD_LO) & (test_income < DEAD_HI)

print(f"\nLower Dead Zone (${DEAD_LO:,.0f} < Income < ${DEAD_HI:,.0f}):")
print(f"  Train Rows: {tr_dead.sum():,} | Positives: {y[tr_dead].sum():,} ({y[tr_dead].mean():.2%})")
print(f"  Test Rows:  {te_dead.sum():,}")

# Distribution of our blend predictions in the Dead Zone
dead_test_preds = final_submission_ranks[te_dead]
print(f"\nCurrent Blend Ranks in Dead Zone (should ideally be near 0):")
print(f"  Max Rank:  {dead_test_preds.max():.4f}")
print(f"  Mean Rank: {dead_test_preds.mean():.4f}")
print(f"  Rows > 0.10: {(dead_test_preds > 0.10).sum():,} / {te_dead.sum():,}")

## 7. Grandmaster Checklist for S6E9

Before submitting in the 0.9464+ tier, check off this list:

1. [x] **No Multi-Seed CV Leakage**: Never evaluate multi-seed bagged predictions over a single OOF array.
2. [x] **Tie-Free Ordinal Ranking**: Always use `method="ordinal"` for continuous ranking. Tied ranks silently degrade ROC-AUC.
3. [x] **Neural Probability Smoothing**: A small allocation (3-5%) of RealMLP eliminates stepped tree plateaus without adding noise.
4. [x] **Orthogonal Paradigms**: Blend across architectures (CTBoost GPU + LightGBM + RealMLP), not just random seeds.
5. [x] **Preserve Strong Anchors**: Never dilute a verified 0.94645 anchor below 70-80% unless you have undeniable frozen-CV proof.

---
### Acknowledgements & References
- **megayak / Artificial Idiocy Lab** — [0.94645 Four Feature Views Beat The Blend](https://www.kaggle.com/code/megayak/s6e9-0-94645-four-feature-views-beat-the-blend)
- **Markus.JM / maiernator** — [S6E9 CTBoost Astra baseline](https://www.kaggle.com/code/maiernator/s6e9-ctboost-not-catboost-astra-baseline)
- **yekenot** — [RealMLP PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch)
- **Chris Deotte** — Original Fable generator reconstruction and EDA
- **Starkhushi** — Domain boundary dead zone analysis (#738968)

*If you found this breakdown and the 0.94645 pipeline helpful, an upvote is greatly appreciated! Good luck on the private leaderboard!*